In [1]:
import optuna
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import lightgbm as lgb
from sklearn.model_selection import KFold, StratifiedKFold
import xgboost as xgb
from xgboost import XGBClassifier
from sklearn.metrics import mean_squared_error, roc_auc_score, accuracy_score
from scipy.stats import rankdata

In [2]:
#Load data
DATA_DIR = '/kaggle/input/competitions/playground-series-s6e5'
train = pd.read_csv(f'{DATA_DIR}/train.csv')
test = pd.read_csv(f'{DATA_DIR}/test.csv')
sample = pd.read_csv(f'{DATA_DIR}/sample_submission.csv')

#Read sample_submission columns
ID_COL = sample.columns[0]
TARGET_COL = sample.columns[1]
print(f'ID column:{ID_COL}')
print(f'TARGET column:{TARGET_COL}')
print(f'train:{train.shape}, test:{test.shape}')

ID column:id
TARGET column:PitNextLap
train:(439140, 16), test:(188165, 15)


In [3]:
pd.set_option('display.max_columns',100)

In [4]:
#EDA
print(train.head())
print(train.dtypes)

   id Driver Compound                   Race  Year  PitStop  LapNumber  Stint  \
0   0   D109     HARD    Canadian Grand Prix  2022        0         50      2   
1   1   D086     HARD       Dutch Grand Prix  2025        1         27      2   
2   2    ZON     HARD    Austrian Grand Prix  2022        0         59      3   
3   3    SPE   MEDIUM     Pre-Season Testing  2023        0          2      1   
4   4   D019     HARD  Azerbaijan Grand Prix  2022        1         26      3   

   TyreLife  Position  LapTime (s)  LapTime_Delta  Cumulative_Degradation  \
0      39.0         8       78.491         -7.564                  21.019   
1       7.0         4       75.095        -32.617                -223.207   
2      22.0        13       70.945         -7.540                -100.529   
3       2.0         7       94.361         -7.324                  -7.324   
4       6.0         2      107.878          8.965                 -14.139   

   RaceProgress  Position_Change  PitNextLap  
0  

In [5]:
# Check missing values
miss = train.isna().sum()
print('Columns with missing values:')
print(miss[miss > 0] if (miss>0).any() else None)

# The distribution of target
print(f'\ntarget[TARGET_COL]:')
print(train[TARGET_COL].describe())
print(f'The number of unique values:{train[TARGET_COL].nunique()}')

Columns with missing values:
None

target[TARGET_COL]:
count    439140.000000
mean          0.198982
std           0.399235
min           0.000000
25%           0.000000
50%           0.000000
75%           0.000000
max           1.000000
Name: PitNextLap, dtype: float64
The number of unique values:2


In [6]:
# Setting
PROBLEM_TYPE ='classification'
N_FOLDS = 5
SEED = 2026

In [7]:
# Feature Engineering
features = [c for c in train.columns if c not in (ID_COL, TARGET_COL, "Driver")]

# Extract object datatype
cat_features = [c for c in features if train[c].dtype == 'object']
for c in cat_features:
    train[c] = train[c].astype('category')
    test[c] = test[c].astype('category')
X = train[features]
y = train[TARGET_COL]
X_test = test[features]

print(f'Number of features: {len(features)}, Number of categories: {len(cat_features)}')
print('Categorial features:',cat_features if cat_features else None)
print(pd.crosstab(train['PitStop'], train['PitNextLap']))

Number of features: 13, Number of categories: 2
Categorial features: ['Compound', 'Race']
PitNextLap     0.0    1.0
PitStop                  
0           306798  72567
1            44961  14814


In [8]:
# CV

# def objective(trial):
#     params = {
#         'objective': 'binary',
#         'metric': 'auc',
#         'verbose': -1,
#         'seed': SEED,
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
#         'num_leaves': trial.suggest_int('num_leaves', 15, 255),
#         'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
#     }
#     oof = np.zeros(len(train))
#     skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
#     for tr_idx, va_idx in skf.split(X, y):
#         X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
#         y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

#         model = lgb.LGBMClassifier(**params, n_estimators=2000)
#         model.fit(X_tr, y_tr,
#                   eval_set=[(X_va, y_va)],
#                   callbacks=[lgb.early_stopping(100), lgb.log_evaluation(0)])
#         oof[va_idx] = model.predict_proba(X_va)[:, 1]

#     return roc_auc_score(y, oof)

# study = optuna.create_study(direction='maximize')   
# study.optimize(objective, n_trials=30)    

# print('最佳 CV AUC:', study.best_value)
# print('最佳参数:', study.best_params)

params = {
     'objective': 'binary',
     'metric':'auc',
     'learning_rate': 0.017545676760792675,
     'num_leaves':185,
     'verbose':-1,
     'min_child_samples': 59, 
     'subsample': 0.888139626704766, 
     'colsample_bytree': 0.6003369973742873, 
     'reg_alpha': 5.088824093962249, 
     'reg_lambda': 0.02751805205560768,
     'seed':SEED,
 }

#out-of-fold prediction
oof_lgb = np.zeros(len(train))  #out-of-fold prediction
preds_lgb = np.zeros(len(test))

splitter = StratifiedKFold(n_splits = N_FOLDS, shuffle = True, random_state = SEED)
split_iter = splitter.split(X,y)

for fold, (tr_idx, va_idx) in enumerate(split_iter):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model = lgb.LGBMClassifier(**params, n_estimators=2000)
    model.fit(
        X_tr, y_tr,
        eval_set = [(X_va, y_va)],
        callbacks = [lgb.early_stopping(100), lgb.log_evaluation(0)]
    )

    oof_lgb[va_idx] = model.predict_proba(X_va)[:,1]
    preds_lgb += model.predict_proba(X_test)[:,1] / N_FOLDS

    print(f'Fold {fold + 1 } finished')

Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1358]	valid_0's auc: 0.94918
Fold 1 finished
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1614]	valid_0's auc: 0.95047
Fold 2 finished
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1619]	valid_0's auc: 0.9505
Fold 3 finished
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1777]	valid_0's auc: 0.951075
Fold 4 finished
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1548]	valid_0's auc: 0.950981
Fold 5 finished


[I 2026-05-23 04:07:39,618] A new study created in memory with name: no-name-6a621f17-0a07-436a-b684-63e5258c5edc
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[838]	valid_0's auc: 0.950406
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[866]	valid_0's auc: 0.948461
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[869]	valid_0's auc: 0.949616
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1223]	valid_0's auc: 0.949062
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1043]	valid_0's auc: 0.949654
[I 2026-05-23 04:10:38,141] Trial 0 finished with value: 0.9494253762568864 and parameters: {'learning_rate': 0.05019584192413463, 'num_leaves': 85, 'min_child_samples': 15, 'subsample': 0.6268887432669653, 'colsample_bytree': 0.7954684049839499, 'reg_alpha': 0.005207877107204066, 'reg_lambda': 1.1676400894145502}. Best is trial 0 with value: 0.9494253762568864.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[351]	valid_0's auc: 0.95048
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[224]	valid_0's auc: 0.948512
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[300]	valid_0's auc: 0.949395
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[286]	valid_0's auc: 0.948529
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[306]	valid_0's auc: 0.949714
[I 2026-05-23 04:12:09,284] Trial 1 finished with value: 0.9493231996922785 and parameters: {'learning_rate': 0.056393920419288146, 'num_leaves': 239, 'min_child_samples': 97, 'subsample': 0.7816839628046579, 'colsample_bytree': 0.8479806405989055, 'reg_alpha': 0.0012306654053101922, 'reg_lambda': 0.00013304960606424257}. Best is trial 0 with value: 0.9494253762568864.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[513]	valid_0's auc: 0.950302
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[411]	valid_0's auc: 0.948126
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[594]	valid_0's auc: 0.949139
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[575]	valid_0's auc: 0.948285
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[404]	valid_0's auc: 0.949402
[I 2026-05-23 04:14:22,841] Trial 2 finished with value: 0.9490395271831014 and parameters: {'learning_rate': 0.037505501157981576, 'num_leaves': 230, 'min_child_samples': 26, 'subsample': 0.7625551433728053, 'colsample_bytree': 0.940746129897071, 'reg_alpha': 0.006450040561950807, 'reg_lambda': 1.8921247224785519e-07}. Best is trial 0 with value: 0.9494253762568864.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[737]	valid_0's auc: 0.950083
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[641]	valid_0's auc: 0.948043
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[540]	valid_0's auc: 0.948954
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[828]	valid_0's auc: 0.948384
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[651]	valid_0's auc: 0.949149
[I 2026-05-23 04:16:41,033] Trial 3 finished with value: 0.9489134786000625 and parameters: {'learning_rate': 0.04465870588844935, 'num_leaves': 129, 'min_child_samples': 20, 'subsample': 0.6251494962150665, 'colsample_bytree': 0.9149887445076162, 'reg_alpha': 6.652824964481858e-08, 'reg_lambda': 6.935839663999077e-06}. Best is trial 0 with value: 0.9494253762568864.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[929]	valid_0's auc: 0.950728
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[903]	valid_0's auc: 0.94883
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[796]	valid_0's auc: 0.949724
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1197]	valid_0's auc: 0.949024
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[903]	valid_0's auc: 0.949887
[I 2026-05-23 04:20:24,265] Trial 4 finished with value: 0.9496251826424964 and parameters: {'learning_rate': 0.02709511586785772, 'num_leaves': 164, 'min_child_samples': 76, 'subsample': 0.7059187390773325, 'colsample_bytree': 0.7422406281012534, 'reg_alpha': 0.0050360817915277994, 'reg_lambda': 2.667594322713081e-06}. Best is trial 4 with value: 0.9496251826424964.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[183]	valid_0's auc: 0.950316
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[195]	valid_0's auc: 0.947963
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[183]	valid_0's auc: 0.948973
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[211]	valid_0's auc: 0.948071
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[247]	valid_0's auc: 0.949469
[I 2026-05-23 04:21:31,212] Trial 5 finished with value: 0.9489441096594069 and parameters: {'learning_rate': 0.07942986435794278, 'num_leaves': 246, 'min_child_samples': 38, 'subsample': 0.9956128779960914, 'colsample_bytree': 0.7546384041635699, 'reg_alpha': 8.592563281616257e-07, 'reg_lambda': 8.184337542676777e-06}. Best is trial 4 with value: 0.9496251826424964.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.948857
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.947055
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.948143
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.947196
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.948034
[I 2026-05-23 04:27:30,401] Trial 6 finished with value: 0.9478509163462966 and parameters: {'learning_rate': 0.010397452325067077, 'num_leaves': 45, 'min_child_samples': 89, 'subsample': 0.9915950587404937, 'colsample_bytree': 0.687573521704894, 'reg_alpha': 0.15140461335500557, 'reg_lambda': 1.0007440182514497e-07}. Best is trial 4 with value: 0.9496251826424964.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[155]	valid_0's auc: 0.950065
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[218]	valid_0's auc: 0.94828
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[205]	valid_0's auc: 0.94917
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[206]	valid_0's auc: 0.948268
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[224]	valid_0's auc: 0.949524
[I 2026-05-23 04:28:39,187] Trial 7 finished with value: 0.949045946633881 and parameters: {'learning_rate': 0.07558940018273602, 'num_leaves': 234, 'min_child_samples': 98, 'subsample': 0.858924543719781, 'colsample_bytree': 0.8133391358725821, 'reg_alpha': 0.0003886633233807918, 'reg_lambda': 2.2497906064385265e-05}. Best is trial 4 with value: 0.9496251826424964.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[372]	valid_0's auc: 0.950156
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[395]	valid_0's auc: 0.94831
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[368]	valid_0's auc: 0.94935
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[484]	valid_0's auc: 0.948487
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[519]	valid_0's auc: 0.949621
[I 2026-05-23 04:30:29,664] Trial 8 finished with value: 0.9491671373992517 and parameters: {'learning_rate': 0.05335742041549571, 'num_leaves': 171, 'min_child_samples': 95, 'subsample': 0.8659753276926108, 'colsample_bytree': 0.9156304207822001, 'reg_alpha': 0.0009541409865414836, 'reg_lambda': 7.5651875899193e-06}. Best is trial 4 with value: 0.9496251826424964.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1196]	valid_0's auc: 0.951053
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1436]	valid_0's auc: 0.94907
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1425]	valid_0's auc: 0.95004
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1356]	valid_0's auc: 0.949341
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1371]	valid_0's auc: 0.950345
[I 2026-05-23 04:36:15,739] Trial 9 finished with value: 0.949960047339514 and parameters: {'learning_rate': 0.015442660853637, 'num_leaves': 208, 'min_child_samples': 51, 'subsample': 0.8247565157260672, 'colsample_bytree': 0.6730303432789645, 'reg_alpha': 0.0009870297027032885, 'reg_lambda': 0.10932274176761338}. Best is trial 9 with value: 0.949960047339514.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1540]	valid_0's auc: 0.951492
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1500]	valid_0's auc: 0.949501
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1577]	valid_0's auc: 0.950261
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1685]	valid_0's auc: 0.949783
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1544]	valid_0's auc: 0.95065
[I 2026-05-23 04:43:33,163] Trial 10 finished with value: 0.9503319482479526 and parameters: {'learning_rate': 0.017545676760792675, 'num_leaves': 185, 'min_child_samples': 59, 'subsample': 0.888139626704766, 'colsample_bytree': 0.6003369973742873, 'reg_alpha': 5.088824093962249, 'reg_lambda': 0.02751805205560768}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1980]	valid_0's auc: 0.951396
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1823]	valid_0's auc: 0.94948
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1880]	valid_0's auc: 0.950355
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1999]	valid_0's auc: 0.949694
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1777]	valid_0's auc: 0.950647
[I 2026-05-23 04:52:36,561] Trial 11 finished with value: 0.9503115176947587 and parameters: {'learning_rate': 0.016554046333310514, 'num_leaves': 182, 'min_child_samples': 60, 'subsample': 0.8912919359738574, 'colsample_bytree': 0.6035596145512832, 'reg_alpha': 7.8178843682822565, 'reg_lambda': 0.06573136931891366}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1461]	valid_0's auc: 0.951403
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1357]	valid_0's auc: 0.94942
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1372]	valid_0's auc: 0.950224
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1709]	valid_0's auc: 0.949675
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1462]	valid_0's auc: 0.950612
[I 2026-05-23 04:59:40,519] Trial 12 finished with value: 0.9502606468454652 and parameters: {'learning_rate': 0.021599335157719796, 'num_leaves': 183, 'min_child_samples': 68, 'subsample': 0.9238412511540338, 'colsample_bytree': 0.6003624765674891, 'reg_alpha': 7.875189128788154, 'reg_lambda': 0.006525588940402363}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1996]	valid_0's auc: 0.951343
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.949478
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.950289
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.949614
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1990]	valid_0's auc: 0.950676
[I 2026-05-23 05:07:17,701] Trial 13 finished with value: 0.9502752654556937 and parameters: {'learning_rate': 0.016726141341683227, 'num_leaves': 125, 'min_child_samples': 58, 'subsample': 0.9157153144185489, 'colsample_bytree': 0.6110892457655206, 'reg_alpha': 5.559075052024948, 'reg_lambda': 0.005271952262775355}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.9513
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.949417
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1997]	valid_0's auc: 0.95027
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1999]	valid_0's auc: 0.949467
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.950404
[I 2026-05-23 05:16:12,484] Trial 14 finished with value: 0.9501677510176388 and parameters: {'learning_rate': 0.010774577953230719, 'num_leaves': 190, 'min_child_samples': 42, 'subsample': 0.9268773442256093, 'colsample_bytree': 0.6600559203594121, 'reg_alpha': 0.2831169590211327, 'reg_lambda': 8.243480500147536}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1952]	valid_0's auc: 0.951206
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1838]	valid_0's auc: 0.949163
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1639]	valid_0's auc: 0.950059
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1784]	valid_0's auc: 0.94942
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1972]	valid_0's auc: 0.950529
[I 2026-05-23 05:22:45,524] Trial 15 finished with value: 0.9500715200294967 and parameters: {'learning_rate': 0.015166242771511233, 'num_leaves': 149, 'min_child_samples': 75, 'subsample': 0.8768807612609443, 'colsample_bytree': 0.6326625167982399, 'reg_alpha': 1.3996589893302863e-05, 'reg_lambda': 0.01264257876574867}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1514]	valid_0's auc: 0.95089
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1576]	valid_0's auc: 0.948993
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1572]	valid_0's auc: 0.949848
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1916]	valid_0's auc: 0.9493
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1632]	valid_0's auc: 0.950221
[I 2026-05-23 05:27:54,187] Trial 16 finished with value: 0.94984208847147 and parameters: {'learning_rate': 0.023720393825447723, 'num_leaves': 97, 'min_child_samples': 58, 'subsample': 0.7271257691358469, 'colsample_bytree': 0.721346771570036, 'reg_alpha': 0.3383277256581602, 'reg_lambda': 0.15361597330651316}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1232]	valid_0's auc: 0.951164
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1037]	valid_0's auc: 0.949091
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1185]	valid_0's auc: 0.950011
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1442]	valid_0's auc: 0.94945
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1465]	valid_0's auc: 0.950472
[I 2026-05-23 05:33:09,022] Trial 17 finished with value: 0.9500292606433272 and parameters: {'learning_rate': 0.019540563642241122, 'num_leaves': 205, 'min_child_samples': 5, 'subsample': 0.9357067626863256, 'colsample_bytree': 0.7061107294958464, 'reg_alpha': 1.2478367976820455, 'reg_lambda': 0.0006256843244837847}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1999]	valid_0's auc: 0.949431
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1997]	valid_0's auc: 0.947194
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.948429
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1999]	valid_0's auc: 0.947554
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.94873
[I 2026-05-23 05:37:32,550] Trial 18 finished with value: 0.9482629776758664 and parameters: {'learning_rate': 0.029657698641240397, 'num_leaves': 27, 'min_child_samples': 44, 'subsample': 0.8225865217012355, 'colsample_bytree': 0.9880431463892994, 'reg_alpha': 0.03906150745062111, 'reg_lambda': 0.10650022725123034}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.950953
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1999]	valid_0's auc: 0.949001
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1993]	valid_0's auc: 0.949964
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.94918
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.950156
[I 2026-05-23 05:43:58,776] Trial 19 finished with value: 0.9498469296317185 and parameters: {'learning_rate': 0.01372071651701024, 'num_leaves': 106, 'min_child_samples': 80, 'subsample': 0.888967573806926, 'colsample_bytree': 0.6514392125486361, 'reg_alpha': 2.9759295354332514e-05, 'reg_lambda': 0.0005608832117616277}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1998]	valid_0's auc: 0.95108
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1999]	valid_0's auc: 0.949285
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1983]	valid_0's auc: 0.950131
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.949287
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1994]	valid_0's auc: 0.950277
[I 2026-05-23 05:52:19,681] Trial 20 finished with value: 0.950008252481086 and parameters: {'learning_rate': 0.013000529917930834, 'num_leaves': 149, 'min_child_samples': 67, 'subsample': 0.959495937289479, 'colsample_bytree': 0.7721956143888637, 'reg_alpha': 1.7936996228155893, 'reg_lambda': 9.552028220062187}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.951272
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1974]	valid_0's auc: 0.9494
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1995]	valid_0's auc: 0.950319
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1998]	valid_0's auc: 0.949619
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.950658
[I 2026-05-23 05:59:36,964] Trial 21 finished with value: 0.950250555116821 and parameters: {'learning_rate': 0.018009013085598814, 'num_leaves': 117, 'min_child_samples': 58, 'subsample': 0.9065817629517049, 'colsample_bytree': 0.6054761806338135, 'reg_alpha': 6.030447805607664, 'reg_lambda': 0.00938507923087023}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.950784
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1998]	valid_0's auc: 0.948703
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1994]	valid_0's auc: 0.949836
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1998]	valid_0's auc: 0.949008
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1996]	valid_0's auc: 0.949952
[I 2026-05-23 06:05:22,696] Trial 22 finished with value: 0.9496522066384263 and parameters: {'learning_rate': 0.01736197420343602, 'num_leaves': 79, 'min_child_samples': 63, 'subsample': 0.8354266061858848, 'colsample_bytree': 0.6309074870674707, 'reg_alpha': 0.04643911017896257, 'reg_lambda': 0.0025622344978826054}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1464]	valid_0's auc: 0.951353
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1328]	valid_0's auc: 0.949348
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1461]	valid_0's auc: 0.950253
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1532]	valid_0's auc: 0.949586
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1517]	valid_0's auc: 0.950597
[I 2026-05-23 06:11:44,912] Trial 23 finished with value: 0.9502252375629402 and parameters: {'learning_rate': 0.024712042871849228, 'num_leaves': 146, 'min_child_samples': 53, 'subsample': 0.9620356845619884, 'colsample_bytree': 0.6095542697403983, 'reg_alpha': 8.55084895243776, 'reg_lambda': 0.48305021799147696}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1731]	valid_0's auc: 0.951463
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1686]	valid_0's auc: 0.94933
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1726]	valid_0's auc: 0.950233
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1733]	valid_0's auc: 0.949556
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1926]	valid_0's auc: 0.950767
[I 2026-05-23 06:19:06,581] Trial 24 finished with value: 0.9502656983707072 and parameters: {'learning_rate': 0.012741005330302507, 'num_leaves': 210, 'min_child_samples': 34, 'subsample': 0.8990540177830435, 'colsample_bytree': 0.6504829546147596, 'reg_alpha': 0.9484232761001233, 'reg_lambda': 0.05641730289695091}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1568]	valid_0's auc: 0.951159
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1326]	valid_0's auc: 0.949166
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1388]	valid_0's auc: 0.950185
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1807]	valid_0's auc: 0.949447
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1372]	valid_0's auc: 0.950422
[I 2026-05-23 06:24:39,094] Trial 25 finished with value: 0.9500691052566956 and parameters: {'learning_rate': 0.02041833596603283, 'num_leaves': 170, 'min_child_samples': 48, 'subsample': 0.9549680353527277, 'colsample_bytree': 0.6905779999290735, 'reg_alpha': 0.03333198641486944, 'reg_lambda': 1.2969180350377705}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1134]	valid_0's auc: 0.95114
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[925]	valid_0's auc: 0.9492
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1174]	valid_0's auc: 0.95013
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1376]	valid_0's auc: 0.949611
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1006]	valid_0's auc: 0.95042
[I 2026-05-23 06:28:40,934] Trial 26 finished with value: 0.9500927050466876 and parameters: {'learning_rate': 0.03330177152915646, 'num_leaves': 126, 'min_child_samples': 83, 'subsample': 0.8511208149992983, 'colsample_bytree': 0.6306810888326038, 'reg_alpha': 1.6294758139847285, 'reg_lambda': 0.02429559859169296}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1998]	valid_0's auc: 0.950239
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.948316
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.949308
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[2000]	valid_0's auc: 0.948612
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1997]	valid_0's auc: 0.949323
[I 2026-05-23 06:34:21,736] Trial 27 finished with value: 0.9491553551051624 and parameters: {'learning_rate': 0.01646053170919751, 'num_leaves': 67, 'min_child_samples': 69, 'subsample': 0.9057800890479164, 'colsample_bytree': 0.8486083170294689, 'reg_alpha': 0.14741972182846264, 'reg_lambda': 0.0024988076842867722}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1998]	valid_0's auc: 0.951149
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1721]	valid_0's auc: 0.949067
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1885]	valid_0's auc: 0.94998
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1990]	valid_0's auc: 0.949317
Training until validation scores don't improve for 100 rounds
Did not meet early stopping. Best iteration is:
[1996]	valid_0's auc: 0.950294
[I 2026-05-23 06:41:57,232] Trial 28 finished with value: 0.9499590951337241 and parameters: {'learning_rate': 0.011278464530952565, 'num_leaves': 192, 'min_child_samples': 59, 'subsample': 0.8037377045732895, 'colsample_bytree': 0.7190596863698825, 'reg_alpha': 5.9398437354461054e-05, 'reg_lambda': 0.0002941919005572632}. Best is trial 10 with value: 0.9503319482479526.
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1599]	valid_0's auc: 0.95137
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1350]	valid_0's auc: 0.949394
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1463]	valid_0's auc: 0.950145
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1625]	valid_0's auc: 0.949644
Training until validation scores don't improve for 100 rounds
Early stopping, best iteration is:
[1680]	valid_0's auc: 0.950587
[I 2026-05-23 06:48:53,889] Trial 29 finished with value: 0.9502242384105548 and parameters: {'learning_rate': 0.022708918925560435, 'num_leaves': 156, 'min_child_samples': 28, 'subsample': 0.6655372799507757, 'colsample_bytree': 0.6269968385071297, 'reg_alpha': 9.612508142719493, 'reg_lambda': 0.6576078047626897}. Best is trial 10 with value: 0.9503319482479526.
最佳 CV AUC: 0.9503319482479526
最佳参数: {'learning_rate': 0.017545676760792675, 'num_leaves': 185, 'min_child_samples': 59, 'subsample': 0.888139626704766, 'colsample_bytree': 0.6003369973742873, 'reg_alpha': 5.088824093962249, 'reg_lambda': 0.02751805205560768}

In [9]:


# def objective_xgb(trial):
#     params = {
#         'objective': 'binary:logistic',
#         'eval_metric': 'auc',
#         'tree_method': 'hist',
#         'enable_categorical': True,
#         'random_state': SEED,
#         'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.1, log=True),
#         'max_depth': trial.suggest_int('max_depth', 3, 10),
#         'min_child_weight': trial.suggest_int('min_child_weight', 1, 100),
#         'subsample': trial.suggest_float('subsample', 0.6, 1.0),
#         'colsample_bytree': trial.suggest_float('colsample_bytree', 0.6, 1.0),
#         'reg_alpha': trial.suggest_float('reg_alpha', 1e-8, 10.0, log=True),
#         'reg_lambda': trial.suggest_float('reg_lambda', 1e-8, 10.0, log=True),
#     }

#     oof = np.zeros(len(train))
#     skf = StratifiedKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)
#     for tr_idx, va_idx in skf.split(X, y):
#         X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
#         y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

#         model = XGBClassifier(
#             **params,
#             n_estimators=2000,
#             early_stopping_rounds=100,
#         )
#         model.fit(X_tr, y_tr,
#                   eval_set=[(X_va, y_va)],
#                   verbose=100)
#         oof[va_idx] = model.predict_proba(X_va)[:, 1]

#     return roc_auc_score(y, oof)

# study_xgb = optuna.create_study(direction='maximize')
# study_xgb.optimize(objective_xgb, n_trials=30)

# print('XGB 最佳 CV AUC:', study_xgb.best_value)
# print('XGB 最佳参数:', study_xgb.best_params)
# XGB 最佳 CV AUC: 0.9501188802462202
# XGB 最佳参数: {'learning_rate': 0.030121389186581537, '
                    # max_depth': 9, '
                    # min_child_weight': 30, '
                    # subsample': 0.802773093901559, '
                    # colsample_bytree': 0.6521109598025392, 
                    # 'reg_alpha': 1.9852464542078325e-07, 
                    # 'reg_lambda': 4.462167064088758e-07}
# reset oof and preds for XGBoost
oof_xgb = np.zeros(len(train))
preds_xgb = np.zeros(len(test))

splitter = StratifiedKFold(
    n_splits=N_FOLDS,
    shuffle=True,
    random_state=SEED
)

for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y)):
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

    model_xgb = XGBClassifier(
        n_estimators=2000,
        learning_rate=0.030121389186581537,
        max_depth=9,
        min_child_weight=30,
        subsample=0.802773093901559,
        colsample_bytree=0.6521109598025392,
        reg_alpha=1.9852464542078325e-07, 
        reg_lambda= 4.462167064088758e-07,
        eval_metric="auc",
        random_state=SEED,
        tree_method="hist",
        enable_categorical=True,
        callbacks=[
            xgb.callback.EarlyStopping(
                rounds=100,
                save_best=True,
                maximize=True
            )
        ]
    )

    model_xgb.fit(
        X_tr, y_tr,
        eval_set=[(X_va, y_va)],
        verbose=False
    )

    oof_xgb[va_idx] = model_xgb.predict_proba(X_va)[:, 1]
    preds_xgb += model_xgb.predict_proba(X_test)[:, 1] / N_FOLDS

    fold_auc = roc_auc_score(y_va, oof_xgb[va_idx])
    print(f"Fold {fold + 1} AUC: {fold_auc:.5f}")

Fold 1 AUC: 0.94887
Fold 2 AUC: 0.95003
Fold 3 AUC: 0.95024
Fold 4 AUC: 0.95076
Fold 5 AUC: 0.95076


In [10]:
# # Catboost
# from catboost import CatBoostClassifier
# # reset oof and preds for CatBoost
# oof_cat = np.zeros(len(train))
# preds_cat = np.zeros(len(test))

# cat_cols = ["Compound", "Race"]

# splitter = StratifiedKFold(
#     n_splits=N_FOLDS,
#     shuffle=True,
#     random_state=SEED
# )

# for fold, (tr_idx, va_idx) in enumerate(splitter.split(X, y)):
#     X_tr, X_va = X.iloc[tr_idx].copy(), X.iloc[va_idx].copy()
#     y_tr, y_va = y.iloc[tr_idx], y.iloc[va_idx]

#     model_cat = CatBoostClassifier(
#         iterations=2000,
#         learning_rate=0.05,
#         depth=6,
#         loss_function="Logloss",
#         eval_metric="AUC",
#         random_seed=SEED,
#         verbose=0,
#         early_stopping_rounds=100
#     )

#     model_cat.fit(
#         X_tr, y_tr,
#         eval_set=(X_va, y_va),
#         cat_features=cat_cols
#     )

#     oof_cat[va_idx] = model_cat.predict_proba(X_va)[:, 1]
#     preds_cat += model_cat.predict_proba(X_test)[:, 1] / N_FOLDS

#     fold_auc = roc_auc_score(y_va, oof_cat[va_idx])
#     print(f"Fold {fold + 1} CatBoost AUC: {fold_auc:.5f}")

In [11]:
# # rank
# lgb_rank = rankdata(oof_lgb)
# xgb_rank = rankdata(oof_xgb)

# rank_blend_oof = 0.6 * lgb_rank + 0.4 * xgb_rank
# print('rank  blend CV:', roc_auc_score(y, rank_blend_oof))

In [12]:
print("OOF LightGBM AUC:", roc_auc_score(y, oof_lgb))
print("OOF XGBoost AUC:", roc_auc_score(y, oof_xgb))
#print("OOF CatBoost AUC:", roc_auc_score(y, oof_cat))
#for wc in [0, 0.1, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 1.0]:
#    for wc_lgb in [0, 0.1, 0.2, 0.3, 0.4, 0.5]:
#    wc_xgb =  1 - wc
#blend = w_other*oof_cat + w_other*oof_lgb
#        blend = wc_lgb*oof_lgb + wc_xgb*oof_xgb + wc_cat*oof_cat
blend = 0.6 *oof_lgb + 0.4 *oof_xgb
#        print(f'Cat weight = {wc_cat}, XGB weight = {wc_xgb}, LGB weight = {wc_lgb}: CV {roc_auc_score(y, blend):.6f}')
print(f'CV : {roc_auc_score(y, blend)}')
        #print(f'cat weight {wc}: CV {roc_auc_score(y, blend):.6f}')
#np.corrcoef( oof_lgb, oof_cat)

OOF LightGBM AUC: 0.9504403683681442
OOF XGBoost AUC: 0.9501304409185438
CV : 0.9505631211570349


## 0.5/0.5
OOF LightGBM AUC: 0.9503273009579485
OOF XGBoost AUC: 0.9501188802462202
CV : 0.9504698993382967

## 0.6/0.4
OOF LightGBM AUC: 0.9503273009579485
OOF XGBoost AUC: 0.9501188802462202
CV : 0.9504807144609456

## SEED 2026
OOF LightGBM AUC: 0.9504403683681442
OOF XGBoost AUC: 0.9501304409185438
CV : 0.9505631211570349

In [13]:
gain = pd.DataFrame({
    'feature': features,
    'gain': model.booster_.feature_importance(importance_type='gain')
}).sort_values('gain', ascending=False)
print(gain.head(15))

                   feature          gain
2                     Year  1.301231e+06
5                    Stint  9.025063e+05
6                 TyreLife  7.148783e+05
1                     Race  5.972372e+05
9            LapTime_Delta  5.236306e+05
4                LapNumber  4.355547e+05
11            RaceProgress  3.705765e+05
12         Position_Change  2.956832e+05
10  Cumulative_Degradation  2.748874e+05
0                 Compound  2.718267e+05
8              LapTime (s)  1.808015e+05
7                 Position  1.239872e+05
3                  PitStop  7.843244e+04


In [14]:
importance = pd.DataFrame({
    'feature': features,
    'importance': model.feature_importances_,
}).sort_values('importance', ascending=False)
print(importance.head(15))

                   feature  importance
10  Cumulative_Degradation       42161
9            LapTime_Delta       38467
8              LapTime (s)       36069
11            RaceProgress       35116
1                     Race       25909
7                 Position       23260
12         Position_Change       22818
4                LapNumber       21129
6                 TyreLife       21098
2                     Year        7832
5                    Stint        4897
0                 Compound        4356
3                  PitStop        1720


In [15]:
# preds_blend = 0.3 * preds_lgb + 0.6 * preds_xgb + 0.1 * preds_cat
preds_blend = 0.6 * preds_lgb + 0.4 * preds_xgb
submission = sample.copy()
submission[TARGET_COL] = preds_blend
submission.to_csv('submission.csv', index=False)
submission.head()

,id,PitNextLap
0,439140,0.003964
1,439141,0.003768
2,439142,0.002904
3,439143,0.195802
4,439144,0.893096
